# Week 3 · Notebook 03 — Fit a Transit, Measure a Planet
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/03_fit_transit_radius.ipynb)

**Goal:** go from "there is a dip" to **"this planet is 1.5 ± 0.0X Earth radii."** You will fit a
model, propagate errors, and compare with the published value.

**Reading:**
- Winn 2010, *Transits and Occultations* ([arXiv:1001.2010](https://arxiv.org/abs/1001.2010)) — Sec. 2 (the geometry)
- Seager & Mallén-Ornelas 2003 ([arXiv:astro-ph/0206228](https://arxiv.org/abs/astro-ph/0206228)) — what a transit shape tells you
- Kreidberg 2015, *batman* ([arXiv:1507.08285](https://arxiv.org/abs/1507.08285)) and its [docs](https://lkreidberg.github.io/batman/docs/html/index.html)

In [ ]:
# --- Setup: run this cell first. Works in Google Colab and locally. ---
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})
print("ready, repo root:", ROOT)

In [ ]:
from transitpipe import io, detrend, search, fit, plotting, utils
STAR = {"rstar": 1.065, "rstar_err": 0.009, "mstar": 0.910, "mstar_err": 0.021}
lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3, 4])
lc.meta.update(STAR)
flat = detrend.sigma_clip(detrend.flatten(lc, window_days=0.5))
res = search.bls(flat, max_period=10)
print(res.summary())

## 1. First, the 10-second estimate
$R_p = \sqrt{\delta}\,R_\star$ using the BLS box depth. Write it down — then see how much better we can do.

In [ ]:
rp_quick = utils.depth_to_rearth(res.depth, STAR["rstar"])
print(f"Quick estimate from BLS depth: Rp ≈ {rp_quick:.2f} R_earth   (published: 1.47)")

## 2. Re-detrend with the transits hidden (do NOT skip this)
During pass 1 the smoother *saw* the transits and was pulled down by them, so transits look
shallower than they are. Now that we know where they are, mask them and detrend again.

In [ ]:
flat2 = detrend.sigma_clip(detrend.flatten_masked(lc, res.period, res.t0, res.duration, window_days=0.5))
for name, l in [("pass 1", flat), ("pass 2", flat2)]:
    core = detrend.transit_mask(l.time, res.period, res.t0, res.duration, pad=0.5)  # each light curve's own time axis
    print(f"mean in-transit depth, {name}: {(1-l.flux[core].mean())*1e6:.0f} ppm")

## 3. Trapezoid fit
Four parameters: mid-time $t_0$, depth $\delta$, total duration $T_{14}$, ingress $\tau$.
Fitted with least squares; error bars from the covariance matrix.

In [ ]:
ft = fit.fit_single_transit(flat2, res, model="trapezoid")
print(ft.summary())
model = lambda t: fit.trapezoid_model(t, ft.t0, ft.period, ft.depth, ft.duration, ft.extra["ingress"])
plotting.plot_fold(flat2, ft.period, ft.t0, model_func=model, window_hours=5); plt.show()

### Error propagation
$R_p = k R_\star$ with $k = \sqrt{\delta}$, so
$$\left(\frac{\sigma_{R_p}}{R_p}\right)^2 = \left(\frac{\sigma_k}{k}\right)^2 + \left(\frac{\sigma_{R_\star}}{R_\star}\right)^2,\qquad \frac{\sigma_k}{k} = \frac{1}{2}\frac{\sigma_\delta}{\delta}$$
Which term dominates for Kepler-10? For a typical TESS star with a 7% radius error?

In [ ]:
frac_k = ft.rp_rs_err / ft.rp_rs
frac_star = STAR["rstar_err"] / STAR["rstar"]
print(f"fractional error from the light curve: {frac_k:.3%}")
print(f"fractional error from the star:        {frac_star:.3%}")

### Sanity-check the error bar with a bootstrap
Make 200 fake data sets (best model + shuffled residuals), refit each. The spread is an error bar
that needs no calculus.

In [ ]:
depths = fit.bootstrap_depth(flat2, ft, n_boot=100)
print(f"covariance depth error: {ft.depth_err*1e6:.1f} ppm   bootstrap: {depths.std()*1e6:.1f} ppm")
plt.hist(depths * 1e6, bins=25); plt.xlabel("depth [ppm]"); plt.ylabel("count"); plt.show()

## 4. The physical model: batman (Mandel & Agol 2002)
Real stars are **limb-darkened**: brighter in the middle, dimmer at the edge. A planet crossing the
centre blocks brighter light ⇒ dip deeper than $(R_p/R_\star)^2$. A trapezoid can't know this; batman can.

Free parameters: $t_0$, $R_p/R_\star$, $a/R_\star$, impact parameter $b$. At modest SNR,
$b$ and $R_p/R_\star$ are **degenerate** (grazing big planet ≈ central small planet). We break it with a
prior on the **stellar density** from the catalogue — exactly what real papers do.

In [ ]:
rho, rho_err = utils.stellar_density(STAR["rstar"], STAR["mstar"], STAR["rstar_err"], STAR["mstar_err"])
print(f"stellar density prior: {rho:.2f} ± {rho_err:.2f} g/cm³  (Sun = 1.41)")
fb = fit.fit_single_transit(flat2, res, model="batman", rho_star=rho, rho_star_err=rho_err,
                            u=(0.40, 0.26), exptime_days=0.0204)   # Kepler long cadence = 29.4 min
print(fb.summary()); print({k: round(v, 3) if isinstance(v, float) else v for k, v in fb.extra.items()})
e = fb.extra
bm = lambda t: fit.batman_model(t, fb.t0, fb.period, fb.rp_rs, e["a_rs"], e["b"], e["u"], 0.0204)
plotting.plot_fold(flat2, fb.period, fb.t0, model_func=bm, window_hours=5, title="batman fit"); plt.show()

`exptime_days=0.0204` matters: a 30-minute exposure *smears* a ~2-hour transit. batman integrates
the model over each exposure (supersampling). Try removing it and compare $R_p/R_\star$ and $b$.

**Limb darkening `u`:** the values above are roughly right for a Sun-like star in the Kepler band.
For your star, look them up in Claret (2011, Kepler) or Claret (2017, TESS) using its $T_\text{eff}$ and $\log g$
(VizieR tables, linked in the project guide).

### ✏️ Your turn
1. Fill in this table for Kepler-10 b **and** your target:

| | quick BLS | trapezoid | batman | published |
|---|---|---|---|---|
| $R_p$ [R⊕] | | | | |

2. Is your batman radius consistent with the published value within the error bars? If not, list
   three possible reasons (think: limb darkening, stellar radius source, detrending, dilution by
   nearby stars — TESS pixels are 21″ wide!).

### 🚀 Going further (full-timers): MCMC for honest error bars
Covariance errors assume a well-behaved, Gaussian likelihood. MCMC maps the actual posterior.
Install `emcee` and `corner`, set `RUN_MCMC = True`. Read: Foreman-Mackey et al. 2013 ([arXiv:1202.3665](https://arxiv.org/abs/1202.3665))
and Hogg & Foreman-Mackey 2018, *Data analysis recipes: MCMC* ([arXiv:1710.06068](https://arxiv.org/abs/1710.06068)).

In [ ]:
RUN_MCMC = False   # set True after `pip install emcee corner`
if RUN_MCMC:
    import emcee, corner
    t, f = flat2.time, flat2.flux
    near = np.abs(((t - fb.t0 + 0.5*fb.period) % fb.period) - 0.5*fb.period) < 3*fb.duration
    t, f = t[near], f[near]
    sig = 1.4826 * np.median(np.abs(f - np.median(f)))

    def log_prob(p):
        t0, k, a, b = p
        if not (0 < k < 0.3 and 1.5 < a < 100 and 0 <= b < 1 + k):
            return -np.inf
        m = fit.batman_model(t, t0, fb.period, k, a, b, e["u"], 0.0204)
        rho_m = 3*np.pi*a**3 / (6.674e-8 * (fb.period*86400)**2)
        return -0.5*np.sum(((f - m)/sig)**2) - 0.5*((rho_m - rho)/rho_err)**2

    ndim, nw = 4, 24
    p0 = np.array([fb.t0, fb.rp_rs, e["a_rs"], min(e["b"], 0.9)]) + 1e-4*np.random.randn(nw, ndim)
    sampler = emcee.EnsembleSampler(nw, ndim, log_prob)
    sampler.run_mcmc(p0, 3000, progress=True)
    chain = sampler.get_chain(discard=1000, thin=10, flat=True)
    corner.corner(chain, labels=["t0", "Rp/Rs", "a/Rs", "b"]); plt.show()
    k_s = chain[:, 1] * STAR["rstar"] * utils.R_SUN_IN_REARTH
    print(f"Rp = {np.median(k_s):.3f} +{np.percentile(k_s,84)-np.median(k_s):.3f} "
          f"-{np.median(k_s)-np.percentile(k_s,16):.3f} R_earth (stellar radius error not included)")

### ✅ Week 3 Checkpoint
- [ ] Kepler-10 b radius with error bar, compared to the published 1.47 R⊕
- [ ] Your target's radius with error bar + the comparison table
- [ ] Draft of the **Methods** section of your report (data → detrend → BLS → fit), ~1 page